In [2]:
import glob
import pandas as pd
import numpy as npp
import matplotlib.pyplot as plt
import os
import unicodedata
from datetime import datetime
import re

In [3]:
path = "/home/hasyim/final_project/Repo-Final-Project-Kelompok-3/data/raw"

In [4]:
data_path = glob.glob(f"{path}/*.csv")

data list:
1. Reviews
2. Order items
3. Product category
4. Cutomers
5. Sellers
6. Orders
7. Products
8. Geolocation
9. Payments

In [5]:
data = []
miss = []
for i in data_path:
    buka_data = pd.read_csv(i)
    missing = buka_data.isna().sum()
    data.append(buka_data)
    miss.append(missing)

In [12]:
data[8].dtypes
kol_waktu_order = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

In [13]:
data[8]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


In [10]:
data[5]['order_estimated_delivery_date'] = data[5]['order_estimated_delivery_date'].apply(pd.to_datetime, format='%Y-%m-%d %H:%M:%S')


In [ ]:
kol_waktu_order = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
data[5][kol_waktu_order] = data[5][kol_waktu_order].apply(pd.to_datetime, format='%Y-%m-%d %H:%M:%S')
data[5].to_csv('/home/hasyim/final_project/Repo-Final-Project-Kelompok-3/data/process/olist_orders_dataset_clean.csv', index=False)

In [11]:
len(data[6][data[6]['product_weight_g']>10000])

1891

In [62]:
def normalize(data):
    if not isinstance(data, str):
        return data
    kata = unicodedata.normalize('NFKD', data)
    kata_normal = ''.join([i for i in kata if not unicodedata.combining(i)])
    return kata_normal

In [19]:
data[7]['geolocation_city'] = data[7]['geolocation_city'].apply(normalize)

In [47]:
data[0].columns
kol_tanggal = ['review_creation_date', 'review_answer_timestamp']
data[0][kol_tanggal] = data[0][kol_tanggal].apply(pd.to_datetime, format='%Y-%m-%d %H:%M:%S')

In [66]:
data[0].columns

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='str')

In [68]:
data_review = ['review_comment_title', 'review_comment_message']
for i in data_review:
    data[0][i] = data[0][i].apply(normalize)
# data[0]['review_comment_message'] = data[0]['review_comment_message'].apply(normalize,na_action='ignore' )

In [69]:
data[0].to_csv('/home/hasyim/final_project/Repo-Final-Project-Kelompok-3/data/process/olist_review.csv', index=False)

In [33]:
data[7].to_csv('/home/hasyim/final_project/Repo-Final-Project-Kelompok-3/data/process/olist_geolocation_dataset.csv', index=False)

In [20]:
loc_fix = pd.read_csv('/home/hasyim/final_project/Repo-Final-Project-Kelompok-3/data/process/olist_geolocation_dataset.csv')

In [21]:
lokasi_zip = loc_fix.drop(columns=['geolocation_lat', 'geolocation_lng', 'geolocation_state']).set_index('geolocation_zip_code_prefix')['geolocation_city'].to_dict()

In [22]:
coba = data[4].copy()

In [14]:
coba

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


In [23]:
coba['seller_city_fix'] = coba['seller_zip_code_prefix'].map(lokasi_zip)

In [25]:
coba['seller_city'] = coba['seller_city_fix'].combine_first(coba['seller_city'])

In [26]:
coba2 = coba.drop(columns=["seller_city_fix"])

In [27]:
coba2.to_csv('/home/hasyim/final_project/Repo-Final-Project-Kelompok-3/data/process/olist_sellers_dataset2.csv', index=False)

In [7]:
penjual = pd.read_csv("/home/hasyim/final_project/Repo-Final-Project-Kelompok-3/data/process/olist_sellers_dataset.csv")

In [8]:
penjual

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi-guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


In [17]:
gabgab = penjual.merge(
    data[4],
    on="seller_id",
    how="left"
)

In [21]:
kos = gabgab[gabgab['seller_city_x'].isna()]
kos

,seller_id,seller_zip_code_prefix_x,seller_city_x,seller_state_x,seller_zip_code_prefix_y,seller_city_y,seller_state_y
473,5962468f885ea01a1b6a97a218797b0a,82040,NaN,PR,82040,curitiba,PR
791,2aafae69bf4c41fbd94053d9413e87ee,91901,NaN,RS,91901,porto alegre,RS
1672,2a50b7ee5aebecc6fd0ff9784a4747d6,72580,NaN,DF,72580,brasilia,DF
1931,2e90cb1677d35cfe24eef47d441b7c87,2285,NaN,SP,2285,sao paulo,SP
2182,0b3f27369a4d8df98f7eb91077e438ac,7412,NaN,SP,7412,aruja,SP
2986,42bde9fef835393bb8a8849cb6b7f245,71551,NaN,DF,71551,brasilia,DF
3028,870d0118f7a9d85960f29ad89d5d989a,37708,NaN,MG,37708,pocos de caldas,MG


In [10]:
gab1 = penjual.merge(
    data[1],
    on="seller_id",
    how="left"
)

In [13]:
gab2 = gab1.merge(
    data[5],
    on="order_id",
    how="left"
)

In [15]:
toko_na = gab2[gab2['seller_city'].isna()]
toko_na

,seller_id,seller_zip_code_prefix,seller_city,seller_state,order_id,order_item_id,product_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
17528,5962468f885ea01a1b6a97a218797b0a,82040,NaN,PR,0a8724bf3ecd49bce712851bbbecaf35,1,48698ba0cfe847b41430fa00081b5619,2017-10-27 00:35:26,39.00,14.10,89aab649693a7e5995fa951d8d2fcdd7,delivered,2017-10-18 23:24:26,2017-10-20 00:35:26,2017-10-24 20:17:49,2017-11-01 19:13:06,2017-11-09 00:00:00
17529,5962468f885ea01a1b6a97a218797b0a,82040,NaN,PR,124fb051a9788c8feb53429fb1612760,1,94aa4a6893284c0236d6c79c194ef5a6,2017-07-07 17:10:15,149.00,16.48,7595f0de4868abaec15c442659fdace2,delivered,2017-07-01 16:58:53,2017-07-01 17:10:15,2017-07-04 06:23:24,2017-07-11 16:42:08,2017-07-28 00:00:00
17530,5962468f885ea01a1b6a97a218797b0a,82040,NaN,PR,21e28426bf49f15032c93153c7a3e206,1,c1b350ede922e02138912a3192bcc86d,2017-10-25 21:56:39,149.00,16.48,ba503b67e4b3b4faef99f1235920617b,delivered,2017-10-18 21:45:26,2017-10-18 21:56:39,2017-10-24 20:17:49,2017-11-03 16:41:38,2017-11-09 00:00:00
17531,5962468f885ea01a1b6a97a218797b0a,82040,NaN,PR,2d58fd72f5132f237b78fe897e578603,1,fb8848eaa1d50492a9c1e8424a86b316,2017-08-11 12:00:22,149.00,14.79,0b85fe0f29638fecccbd53288d8cf0f7,delivered,2017-08-04 11:00:09,2017-08-04 12:00:22,2017-08-07 21:33:40,2017-08-11 22:58:00,2017-08-29 00:00:00
17532,5962468f885ea01a1b6a97a218797b0a,82040,NaN,PR,8d16656866ce7d67f81d49a9707ab76e,1,48698ba0cfe847b41430fa00081b5619,2018-01-03 18:28:04,39.00,15.10,aa1e003c69c80cdf08ae8225270769b3,invoiced,2017-12-25 18:36:16,2017-12-26 18:28:04,NaN,NaN,2018-01-30 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111595,870d0118f7a9d85960f29ad89d5d989a,37708,NaN,MG,eb0786ebf8a32df3392caaf0727e7bbd,1,6fd3f7378cd65325e10cb21733778416,2018-01-03 04:08:44,89.90,13.65,4ea4534f03b9bec97d923dc02b404656,delivered,2017-12-23 21:25:59,2017-12-27 04:08:44,2017-12-28 16:52:23,2018-01-08 22:25:05,2018-01-22 00:00:00
111596,870d0118f7a9d85960f29ad89d5d989a,37708,NaN,MG,fa0eabdc8b8a7fb4e0e95a2b70dabf8b,1,21025aea6e634ed0ab9709a4fa5fb69c,2017-11-16 18:10:34,38.90,14.08,42669d66e9ebba92f46400f726886844,delivered,2017-11-09 17:54:37,2017-11-09 18:10:34,2017-11-13 19:28:35,2017-11-22 00:48:47,2017-11-30 00:00:00
111597,870d0118f7a9d85960f29ad89d5d989a,37708,NaN,MG,fa0eabdc8b8a7fb4e0e95a2b70dabf8b,2,21025aea6e634ed0ab9709a4fa5fb69c,2017-11-16 18:10:34,38.90,14.08,42669d66e9ebba92f46400f726886844,delivered,2017-11-09 17:54:37,2017-11-09 18:10:34,2017-11-13 19:28:35,2017-11-22 00:48:47,2017-11-30 00:00:00
111598,870d0118f7a9d85960f29ad89d5d989a,37708,NaN,MG,faa0a8e07c82b4188dac82ef1d170979,1,f94d494034180bcbeb9c56cae79f6168,2017-12-04 03:47:29,47.90,19.21,9be47708361b959a33fb0b2a537c02ef,delivered,2017-11-25 00:23:59,2017-11-28 04:31:17,2017-11-30 19:28:54,2017-12-15 19:51:54,2017-12-19 00:00:00


In [40]:
data[4]['kota_clean'] = coba['seller_city']
data[4][data[4]['seller_city'] != data[4]['kota_clean']]

,seller_id,seller_zip_code_prefix,seller_city,seller_state,kota_clean
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,mogi-guacu
6,e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE,brejão
19,f9ec7093df3a7b346b7bcf7864069ca3,5138,sao paulo,SP,são paulo
20,4e6015589b781adaa5ce7f1892d06bb1,11440,guaruja,SP,guarujá
28,430315b7bb4b6e4b3c978f9dfa9b0558,4857,sao paulo,SP,são paulo
...,...,...,...,...,...
3026,17e34d8224d27a541263c4c64b11a56b,14085,riberao preto,SP,ribeirao preto
3028,870d0118f7a9d85960f29ad89d5d989a,37708,pocos de caldas,MG,NaN
3044,778323240ce2830d68aab11794e00bfb,13600,sao paulo,SP,araras
3060,21e83881401b92b49fb09a16d3852291,38405,uberlandia,MG,uberlândia


In [15]:
coba = data[5].groupby("order_status")
coba.get_group("canceled").isna().sum()

order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                141
order_delivered_carrier_date     550
order_delivered_customer_date    619
order_estimated_delivery_date      0
dtype: int64

In [24]:
pd.isna(data[0]['review_comment_message'][0])

True

In [17]:
from qdrant_client import QdrantClient
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
url = os.getenv("QDRANT_URL")
qdrant_api = os.getenv("QDRANT_API")

In [12]:
embedding = OpenAIEmbeddings(
    model='text-embedding-3-small',
)

In [30]:
documents=[]
for i in range(len(data[0])):
    review_id = data[0]['review_id'][i]
    order_id = data[0]['order_id'][i]
    review_score = data[0]['review_score'][i]
    review_title = data[0]['review_comment_title'][i]
    review = data[0]['review_comment_message'][i]
    if pd.isna(review):
        pass
    else:
        doc = Document(
            page_content=review,
            metadata={
                "review_id": review_id,
                "order_id": order_id,
                "review_score": int(review_score),
                "review_title": review_title
            },
        )
        documents.append(doc)

In [32]:
from uuid import uuid4
uuids = [str(uuid4()) for _ in range(len(documents))]

In [33]:
qdrant = QdrantVectorStore.from_documents(
    documents,
    embedding,
    url=url,
    api_key=qdrant_api,
    collection_name="olist_data",
)

print("Qdrant sukses simpan data")

KeyboardInterrupt: 